# Sensitivity visualizations

This notebook reads the existing sensitivity-analysis outputs from the HECO study and generates publication-oriented figures without modifying any of the original simulation inputs or results.

In [1]:
from pathlib import Path
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml

try:
    import seaborn as sns
    sns.set_theme(style="ticks", context="talk")
except Exception:
    sns = None

notebook_dir = Path.cwd().resolve()
if not (notebook_dir / "sa_2_2").exists():
    notebook_dir = notebook_dir.parent

sensitivity_dir = notebook_dir
output_dir = sensitivity_dir / "figures_sensitivity"
output_dir.mkdir(exist_ok=True, parents=True)


## 1. Setup and validation

The notebook validates the presence of the relevant metric CSV tables and YAML configurations before plotting. It explicitly reports missing tests and highlights whether test_id 1 is absent from the expected sensitivity sets.

In [4]:
def discover_yaml_ids(yaml_dir: Path, pattern: str):
    files = sorted(yaml_dir.glob(pattern))
    ids = []
    for path in files:
        try:
            stem = path.stem
            ids.append(int(stem.rsplit("_", 1)[-1]))
        except ValueError:
            continue
    return sorted(ids)

def read_metric_table(csv_path: Path):
    table = pd.read_csv(csv_path)
    if "test_id" in table.columns:
        table["test_id"] = pd.to_numeric(table["test_id"], errors="coerce").astype("Int64")
    return table

datasets = {
    "sa_2_2": {
        "yaml_dir": sensitivity_dir / "sa_2_2/input_yaml_files",
        "yaml_pattern": "sa_2_2_test_*.yaml",
        "metric_csv": sensitivity_dir / "sa_2_2/heco_results_test_polygons_metrics.csv",
        "final_csv": sensitivity_dir / "sa_2_2/heco_results_test_polygons_metrics_last_iteration.csv",
        "color": "#ff7f0e",
        "label": "sa_2_2",
    },
    "sa_2_3": {
        "yaml_dir": sensitivity_dir / "sa_2_3/input_yaml_files",
        "yaml_pattern": "sa_2_3_test_*.yaml",
        "metric_csv": sensitivity_dir / "sa_2_3/heco_results_test_polygons_metrics.csv",
        "final_csv": sensitivity_dir / "sa_2_3/heco_results_test_polygons_metrics_last_iteration.csv",
        "color": "#2ca02c",
        "label": "sa_2_3",
    },
    "sa_2_4": {
        "yaml_dir": sensitivity_dir / "sa_2_4/input_yaml_files",
        "yaml_pattern": "sa_2_4_test_*.yaml",
        "metric_csv": sensitivity_dir / "sa_2_4/heco_results_test_polygons_metrics.csv",
        "final_csv": sensitivity_dir / "sa_2_4/heco_results_test_polygons_metrics_last_iteration.csv",
        "color": "#d62728",
        "label": "sa_2_4",
    },
    "sa_2_5": {
        "yaml_dir": sensitivity_dir / "sa_2_5/input_yaml_files",
        "yaml_pattern": "sa_2_5_test_*.yaml",
        "metric_csv": sensitivity_dir / "sa_2_5/heco_results_test_polygons_metrics.csv",
        "final_csv": sensitivity_dir / "sa_2_5/heco_results_test_polygons_metrics_last_iteration.csv",
        "color": "#9467bd",
        "label": "sa_2_5",
    },
}

inventory = {}
for name, spec in datasets.items():
    yaml_dir = spec["yaml_dir"]
    metric_csv = spec["metric_csv"]
    final_csv = spec["final_csv"]
    yaml_exists = yaml_dir.exists()
    csv_exists = metric_csv.exists()
    final_exists = final_csv.exists()
    yaml_ids = discover_yaml_ids(yaml_dir, spec["yaml_pattern"]) if yaml_exists else []
    metric_table = read_metric_table(metric_csv) if csv_exists else pd.DataFrame()
    final_table = read_metric_table(final_csv) if final_exists else pd.DataFrame()
    full_metric_ids = sorted([int(v) for v in metric_table["test_id"].dropna().astype(int).unique().tolist()]) if "test_id" in metric_table.columns else []
    final_metric_ids = sorted([int(v) for v in final_table["test_id"].dropna().astype(int).unique().tolist()]) if "test_id" in final_table.columns else []
    missing_full = [i for i in yaml_ids if i not in full_metric_ids]
    missing_final = [i for i in yaml_ids if i not in final_metric_ids]
    inventory[name] = {
        "yaml_ids": yaml_ids,
        "full_metric_ids": full_metric_ids,
        "final_metric_ids": final_metric_ids,
        "missing_full": missing_full,
        "missing_final": missing_final,
        "yaml_exists": yaml_exists,
        "csv_exists": csv_exists,
        "final_exists": final_exists,
    }
    print(
        f"{name}: yaml_configs={len(yaml_ids)}, full_metric_tests={len(full_metric_ids)}, final_metric_tests={len(final_metric_ids)}, missing_full={missing_full}, missing_final={missing_final}"
    )
    if 1 not in yaml_ids or 1 not in full_metric_ids or 1 not in final_metric_ids:
        print(f"WARNING: test_id 1 is absent or incomplete for {name}.")

assert all(inv["yaml_exists"] and inv["csv_exists"] and inv["final_exists"] for inv in inventory.values()), "One or more required CSV or YAML inputs are missing."

sa_2_2: yaml_configs=100, full_metric_tests=99, final_metric_tests=98, missing_full=[1], missing_final=[0, 1]
sa_2_3: yaml_configs=300, full_metric_tests=299, final_metric_tests=298, missing_full=[1], missing_final=[0, 1]
sa_2_4: yaml_configs=100, full_metric_tests=99, final_metric_tests=98, missing_full=[1], missing_final=[0, 1]
sa_2_5: yaml_configs=100, full_metric_tests=99, final_metric_tests=99, missing_full=[0], missing_final=[0]


## 2. Common definitions

The metric semantics and plotting conventions are defined once for reuse across all panels.

In [5]:
METRICS = {
    "SRA": {"label": "SRA", "better": "higher is better", "cmap": "cividis", "vmin": 0.0, "vmax": 1.0},
    "CI": {"label": "CI", "better": "lower is better", "cmap": "cividis_r", "vmin": 0.0, "vmax": None},
    "Jaccard": {"label": "Jaccard", "better": "higher is better", "cmap": "cividis", "vmin": 0.0, "vmax": 1.0},
    "DICE": {"label": "DICE", "better": "higher is better", "cmap": "cividis", "vmin": 0.0, "vmax": 1.0},
}

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    return 6371.0 * c

def load_yaml_origin(yaml_path: Path):
    with yaml_path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    input_config = config.get("input", {})
    return float(input_config.get("lat0", np.nan)), float(input_config.get("lon0", np.nan))

def resolve_yaml_path(dataset_name: str, test_id: int):
    base = sensitivity_dir / dataset_name
    if dataset_name in {"sa_2_2", "sa_2_3", "sa_2_4", "sa_2_5"}:
        base = sensitivity_dir / dataset_name / "input_yaml_files"
    return base / f"{dataset_name}_test_{test_id}.yaml"

def make_metric_panel(ax, data, metric, extent, label_ids=False, benchmark_origin=None):
    cfg = METRICS[metric]
    values = data[metric].astype(float)
    mask = np.isfinite(values)
    if metric == "CI":
        vmax = np.nanpercentile(values[mask], 95) if mask.any() else 1.0
        vmax = max(vmax, 1e-6)
        vmin = 0.0
    else:
        vmin, vmax = 0.0, 1.0
    sc = ax.scatter(
        data.loc[mask, "lon"],
        data.loc[mask, "lat"],
        c=data.loc[mask, metric],
        cmap=cfg["cmap"],
        vmin=vmin,
        vmax=vmax,
        s=35,
        edgecolors="k",
        linewidths=0.2,
        alpha=0.95,
    )
    if benchmark_origin is not None:
        ax.scatter(
            [benchmark_origin[1]],
            [benchmark_origin[0]],
            marker="*",
            s=260,
            facecolors="none",
            edgecolors="black",
            linewidths=1.6,
            zorder=6,
        )
    if label_ids:
        for _, row in data[data["test_id"] <= 10].iterrows():
            ax.text(row["lon"], row["lat"], str(int(row["test_id"])), fontsize=7, color="0.15")
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(data["lat"].astype(float)))))
    ax.set_title(f"{metric} ({cfg['better']})", fontsize=11)
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    ax.grid(alpha=0.25)
    return sc

def style_axis(ax):
    ax.tick_params(labelsize=8)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

def save_figure(fig, stem, dpi=300):
    fig.savefig(output_dir / f"{stem}.png", dpi=dpi, bbox_inches="tight")
    fig.savefig(output_dir / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)

## 3. Load the origin-perturbation data

Distances are computed from the YAML input coordinates using the haversine formula rather than from output-polygon centroids.

In [6]:
combined_origin_parts = [("sa_2_2", False), ("sa_2_3", False)]
combined_rows = []
benchmark_origin = load_yaml_origin(resolve_yaml_path("sa_2_3", 0))

for dataset_name, label_ids in combined_origin_parts:
    final_table = pd.read_csv(datasets[dataset_name]["final_csv"])
    final_table["test_id"] = pd.to_numeric(final_table["test_id"], errors="coerce").astype("Int64")
    for test_id in sorted(final_table["test_id"].dropna().astype(int).unique()):
        if test_id == 0:
            continue
        yaml_path = resolve_yaml_path(dataset_name, test_id)
        if not yaml_path.exists():
            print(f"Missing YAML for {dataset_name} test {test_id}: {yaml_path}")
            continue
        lat0, lon0 = load_yaml_origin(yaml_path)
        row = final_table[final_table["test_id"] == test_id].iloc[0].to_dict()
        row["lat"] = lat0
        row["lon"] = lon0
        row["distance_km"] = haversine_km(benchmark_origin[0], benchmark_origin[1], lat0, lon0)
        row["part"] = dataset_name
        combined_rows.append(row)

plot_frame = pd.DataFrame(combined_rows)
metric_cols = [m for m in ["SRA", "CI", "Jaccard", "DICE"] if m in plot_frame.columns]
for col in metric_cols:
    plot_frame[col] = pd.to_numeric(plot_frame[col], errors="coerce")
plot_frame["test_id"] = plot_frame["test_id"].astype(int)
origin_results = [{
    "name": "sa_2_2_sa_2_3",
    "label": "sa_2_2 + sa_2_3",
    "color": "#2ca02c",
    "data": plot_frame,
    "benchmark_origin": benchmark_origin,
    "label_ids": False,
}]

for result in origin_results:
    data = result["data"]
    metrics = ["SRA", "CI", "Jaccard", "DICE"]
    extent = (
        data["lon"].min() - 0.02,
        data["lon"].max() + 0.02,
        data["lat"].min() - 0.02,
        data["lat"].max() + 0.02,
    )
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
    fig.suptitle("sa_2_2 + sa_2_3 combined origin perturbation: metric maps", fontsize=14, y=0.98)
    for ax, metric in zip(axes.flat, metrics):
        sc = make_metric_panel(ax, data[["lat", "lon", "test_id", metric]], metric, extent, label_ids=result["label_ids"], benchmark_origin=result["benchmark_origin"])
        style_axis(ax)
    fig.subplots_adjust(wspace=0.20, hspace=0.24, right=0.94)
    cbar_ax = fig.add_axes([0.95, 0.15, 0.02, 0.70])
    fig.colorbar(sc, cax=cbar_ax)
    save_figure(fig, "sa_2_2_sa_2_3_origin_metric_maps")

combined_data = origin_results[0]["data"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)
fig.suptitle("sa_2_2 + sa_2_3 combined origin perturbation: manuscript SRA and CI", fontsize=13)
extent = (combined_data["lon"].min() - 0.02, combined_data["lon"].max() + 0.02, combined_data["lat"].min() - 0.02, combined_data["lat"].max() + 0.02)
for ax, metric in zip(axes, ["SRA", "CI"]):
    sc = make_metric_panel(ax, combined_data[["lat", "lon", "test_id", metric]], metric, extent, label_ids=False, benchmark_origin=origin_results[0]["benchmark_origin"])
    style_axis(ax)
fig.subplots_adjust(wspace=0.20, bottom=0.20)
cbar_ax = fig.add_axes([0.94, 0.18, 0.02, 0.60])
fig.colorbar(sc, cax=cbar_ax)
save_figure(fig, "sa_2_2_sa_2_3_origin_SRA_CI_manuscript")

## 4. Origin-distance response

The notebook compares the final-iteration metrics against the actual origin-perturbation distance using raw observations, binned medians, and interquartile ranges.

In [7]:
from scipy.stats import spearmanr

def summarize_quantile_bins(data, metric, n_bins=6):
    dist = data["distance_km"].astype(float).to_numpy()
    ranks = pd.Series(dist).rank(method="first").to_numpy()
    edges = np.quantile(ranks, np.linspace(0.0, 1.0, n_bins + 1))
    edges[0] = -np.inf
    edges[-1] = np.inf
    bins = np.digitize(ranks, edges[1:-1], right=False)
    rows = []
    for b in np.unique(bins):
        subset = data.iloc[np.where(bins == b)[0]]
        if subset.empty:
            continue
        rows.append({
            "x": subset["distance_km"].median(),
            "median": subset[metric].median(),
            "q25": subset[metric].quantile(0.25),
            "q75": subset[metric].quantile(0.75),
        })
    return pd.DataFrame(rows).sort_values("x")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Combined origin-distance response: final-iteration metrics", fontsize=14)
metrics = ["SRA", "CI", "Jaccard", "DICE"]
for ax, metric in zip(axes.flat, metrics):
    for result in origin_results:
        data = result["data"]
        ax.scatter(
            data["distance_km"],
            data[metric],
            s=20,
            alpha=0.35,
            color=result["color"],
            label=result["label"],
        )
        summary = summarize_quantile_bins(data, metric)
        if not summary.empty:
            ax.plot(summary["x"], summary["median"], color=result["color"], linewidth=2.0)
            ax.fill_between(summary["x"], summary["q25"], summary["q75"], color=result["color"], alpha=0.18)
        corr = spearmanr(data["distance_km"], data[metric])
        ax.text(0.02, 0.95, f"Spearman ρ={corr.statistic:.2f}", transform=ax.transAxes, fontsize=9, va="top")
    ax.set_xlabel("Origin perturbation distance (km)")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.25)
    if metric == "CI":
        ax.set_ylim(0, max(1.0, data[metric].quantile(0.95) * 1.15))
    else:
        ax.set_ylim(0, 1.0)
    ax.set_xlim(0, max(1.0, data["distance_km"].max() * 1.05))
    style_axis(ax)
fig.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.96), frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.94])
save_figure(fig, "origin_distance_response")

combined_response = origin_results[0]["data"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
fig.suptitle("sa_2_2 + sa_2_3 combined manuscript: distance response for SRA and CI", fontsize=13)
for ax, metric in zip(axes, ["SRA", "CI"]):
    data = combined_response
    ax.scatter(data["distance_km"], data[metric], s=20, alpha=0.45, color=origin_results[0]["color"], label="sa_2_2 + sa_2_3")
    summary = summarize_quantile_bins(data, metric)
    if not summary.empty:
        ax.plot(summary["x"], summary["median"], color=origin_results[0]["color"], linewidth=2.0)
        ax.fill_between(summary["x"], summary["q25"], summary["q75"], color=origin_results[0]["color"], alpha=0.18)
    ax.set_xlabel("Origin perturbation distance (km)")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.25)
    ax.set_ylim(0, 1.0 if metric != "CI" else max(1.0, combined_response[metric].quantile(0.95) * 1.15))
    style_axis(ax)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, "sa_2_2_sa_2_3_origin_distance_response_manuscript")

## 5. Diffusion-coefficient sensitivity

The notebook reads the sa_2_4 final-iteration dataset and the full metric table to generate descriptive parameter-response plots and temporal heatmaps.

In [8]:
diffusion_full = pd.read_csv(datasets["sa_2_4"]["metric_csv"])
diffusion_full["test_id"] = pd.to_numeric(diffusion_full["test_id"], errors="coerce").astype("Int64")
diffusion_final = pd.read_csv(datasets["sa_2_4"]["final_csv"])
diffusion_final["test_id"] = pd.to_numeric(diffusion_final["test_id"], errors="coerce").astype("Int64")

parameter_values = []
for test_id in sorted(diffusion_full["test_id"].dropna().astype(int).unique()):
    yaml_path = resolve_yaml_path("sa_2_4", test_id)
    if yaml_path.exists():
        lat0, lon0 = load_yaml_origin(yaml_path)
        with yaml_path.open("r", encoding="utf-8") as handle:
            config = yaml.safe_load(handle)
        parameter_values.append((test_id, float(config["input"].get("sim_diffusion_coeff", np.nan))))
parameter_lookup = dict(parameter_values)

diffusion_full["sim_diffusion_coeff"] = diffusion_full["test_id"].map(parameter_lookup)
diffusion_final["sim_diffusion_coeff"] = diffusion_final["test_id"].map(parameter_lookup)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Diffusion-coefficient response (descriptive)", fontsize=14)
for ax, metric in zip(axes.flat, ["SRA", "CI", "Jaccard", "DICE"]):
    data = diffusion_final[["sim_diffusion_coeff", metric]].dropna()
    ax.scatter(data["sim_diffusion_coeff"], data[metric], s=20, alpha=0.55, color="#1f77b4")
    centered = data[metric].rolling(window=11, center=True, min_periods=11).median()
    ax.plot(data["sim_diffusion_coeff"], centered, color="#d62728", linewidth=2.0, label="11-point centered rolling median (descriptive)")
    ax.axvline(10.0, color="black", linestyle="--", linewidth=1.2)
    ax.set_xlabel("sim_diffusion_coefficient")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.25)
    ax.set_xlim(0.8, 20.0)
    ax.set_ylim(0, 1.0 if metric != "CI" else max(1.0, data[metric].quantile(0.95) * 1.15))
    style_axis(ax)
fig.legend(loc="upper center", ncol=1, bbox_to_anchor=(0.92, 0.97), frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, "diffusion_coefficient_response")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Diffusion-coefficient temporal heatmaps", fontsize=14)
for ax, metric in zip(axes.flat, ["SRA", "CI", "Jaccard", "DICE"]):
    agg = diffusion_full[["test_id", "sim_step", "sim_diffusion_coeff", metric]].copy()
    agg = agg.dropna(subset=[metric, "sim_diffusion_coeff"])
    agg["sim_step"] = pd.to_numeric(agg["sim_step"], errors="coerce").astype(int)
    agg = agg.groupby(["sim_diffusion_coeff", "sim_step"], as_index=False)[metric].mean()
    pivot = agg.pivot(index="sim_diffusion_coeff", columns="sim_step", values=metric).sort_index()
    pivot = pivot.sort_index(axis=0).sort_index(axis=1)
    if metric == "CI":
        vmin, vmax = 0.0, max(1e-6, np.nanpercentile(pivot.values[~np.isnan(pivot.values)], 95))
    else:
        vmin, vmax = 0.0, 1.0
    im = ax.imshow(pivot.values, aspect="auto", cmap=METRICS[metric]["cmap"], vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(metric)
    ax.set_xlabel("LPDM iteration")
    ax.set_ylabel("Diffusion coefficient")
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([str(int(x)) for x in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([f"{x:.2f}" for x in pivot.index], rotation=0)
    style_axis(ax)
    fig.colorbar(im, ax=ax, shrink=0.9, pad=0.03)
fig.tight_layout()
save_figure(fig, "diffusion_coefficient_temporal_heatmaps")

## 6. Spilled-volume diagnostic

The notebook creates the same descriptive response and temporal-heatmap visualizations for the spilled-volume sensitivity experiment, but labels them as diagnostic checks rather than physical sensitivity.

In [9]:
volume_full = pd.read_csv(datasets["sa_2_5"]["metric_csv"])
volume_full["test_id"] = pd.to_numeric(volume_full["test_id"], errors="coerce").astype("Int64")
volume_final = pd.read_csv(datasets["sa_2_5"]["final_csv"])
volume_final["test_id"] = pd.to_numeric(volume_final["test_id"], errors="coerce").astype("Int64")

parameter_values = []
for test_id in sorted(volume_full["test_id"].dropna().astype(int).unique()):
    yaml_path = resolve_yaml_path("sa_2_5", test_id)
    if yaml_path.exists():
        with yaml_path.open("r", encoding="utf-8") as handle:
            config = yaml.safe_load(handle)
        parameter_values.append((test_id, float(config["input"].get("volume_spilled_m3", np.nan))))
parameter_lookup = dict(parameter_values)

volume_full["volume_spilled_m3"] = volume_full["test_id"].map(parameter_lookup)
volume_final["volume_spilled_m3"] = volume_final["test_id"].map(parameter_lookup)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Spilled-volume diagnostic response (invariance check)", fontsize=14)
for ax, metric in zip(axes.flat, ["SRA", "CI", "Jaccard", "DICE"]):
    data = volume_final[["volume_spilled_m3", metric]].dropna()
    ax.scatter(data["volume_spilled_m3"], data[metric], s=20, alpha=0.55, color="#9467bd")
    centered = data[metric].rolling(window=11, center=True, min_periods=11).median()
    ax.plot(data["volume_spilled_m3"], centered, color="#d62728", linewidth=2.0, label="11-point centered rolling median (descriptive)")
    ax.set_xlabel("volume_spilled_m3")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.25)
    ax.set_xscale("log")
    ax.set_ylim(0, 1.0 if metric != "CI" else max(1.0, data[metric].quantile(0.95) * 1.15))
    style_axis(ax)
fig.legend(loc="upper center", ncol=1, bbox_to_anchor=(0.92, 0.97), frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, "spilled_volume_diagnostic_response")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Spilled-volume temporal heatmaps (diagnostic)", fontsize=14)
for ax, metric in zip(axes.flat, ["SRA", "CI", "Jaccard", "DICE"]):
    agg = volume_full[["test_id", "sim_step", "volume_spilled_m3", metric]].copy()
    agg = agg.dropna(subset=[metric, "volume_spilled_m3"])
    agg["sim_step"] = pd.to_numeric(agg["sim_step"], errors="coerce").astype(int)
    agg = agg.groupby(["volume_spilled_m3", "sim_step"], as_index=False)[metric].mean()
    pivot = agg.pivot(index="volume_spilled_m3", columns="sim_step", values=metric).sort_index()
    pivot = pivot.sort_index(axis=0).sort_index(axis=1)
    if metric == "CI":
        vmin, vmax = 0.0, max(1e-6, np.nanpercentile(pivot.values[~np.isnan(pivot.values)], 95))
    else:
        vmin, vmax = 0.0, 1.0
    im = ax.imshow(pivot.values, aspect="auto", cmap=METRICS[metric]["cmap"], vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(metric)
    ax.set_xlabel("LPDM iteration")
    ax.set_ylabel("Volume spilled (m^3)")
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([str(int(x)) for x in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([f"{x:.2f}" for x in pivot.index], rotation=0)
    style_axis(ax)
    fig.colorbar(im, ax=ax, shrink=0.9, pad=0.03)
fig.tight_layout()
save_figure(fig, "spilled_volume_temporal_heatmaps")

## 8. Interpretation summary

- Spatial maps are effective for origin perturbations because the location of the release is explicit and the changes are naturally visualized in geographic space.
- Distance-response plots complement the maps by showing whether the metric changes smoothly with the actual perturbation magnitude.
- SRA should not be interpreted alone because it can remain high even when the spatial pattern is displaced or fragmented.
- Jaccard and Dice are visually redundant because Dice is a monotonic transformation of Jaccard.
- The 300-location origin-perturbation set is the strongest candidate for the manuscript because it resolves spatial structure more clearly and provides better coverage of the origin domain.
- Volume plots should remain diagnostic because the examined volume parameter is not expected to alter the particle-displacement physics directly.

In [10]:
generated_files = sorted([p for p in output_dir.glob("*") if p.is_file() and p.suffix.lower() in {".png", ".pdf"}])
for path in generated_files:
    print(path.relative_to(sensitivity_dir))
print(f"Generated {len(generated_files)} figure files in {output_dir}.")

figures_sensitivity/._diffusion_coefficient_response.pdf
figures_sensitivity/._diffusion_coefficient_response.png
figures_sensitivity/._diffusion_coefficient_temporal_heatmaps.pdf
figures_sensitivity/._diffusion_coefficient_temporal_heatmaps.png
figures_sensitivity/._origin_distance_response.pdf
figures_sensitivity/._origin_distance_response.png
figures_sensitivity/._sa_2_2_sa_2_3_origin_SRA_CI_manuscript.pdf
figures_sensitivity/._sa_2_2_sa_2_3_origin_SRA_CI_manuscript.png
figures_sensitivity/._sa_2_2_sa_2_3_origin_distance_response_manuscript.pdf
figures_sensitivity/._sa_2_2_sa_2_3_origin_distance_response_manuscript.png
figures_sensitivity/._sa_2_2_sa_2_3_origin_metric_maps.pdf
figures_sensitivity/._sa_2_2_sa_2_3_origin_metric_maps.png
figures_sensitivity/._spilled_volume_diagnostic_response.pdf
figures_sensitivity/._spilled_volume_diagnostic_response.png
figures_sensitivity/._spilled_volume_temporal_heatmaps.pdf
figures_sensitivity/._spilled_volume_temporal_heatmaps.png
figures_sens